# Day 2: Streaming Ingestion (Kafka / MSK + Spark Streaming)

This notebook simulates streaming ingestion of synthetic FHIR Observation & Patient streams via PySpark memory streams into the Bronze table.

In [ ]:
!pip install pyspark pyiceberg
import os
import json
import tempfile

# Set Spark version if needed
os.environ['SPARK_VERSION'] = '3.5'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType

# Initialize Spark Session with Iceberg configurations
spark = SparkSession.builder \
    .appName("Day02_Streaming_Simulation") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

print("Spark Session initialized.")

In [ ]:
# Create Bronze database and table (if not exists from Day 1)
spark.sql("CREATE DATABASE IF NOT EXISTS local.ehdip_bronze_db")

spark.sql("""
CREATE TABLE IF NOT EXISTS local.ehdip_bronze_db.raw_payloads (
    payload_id STRING,
    source_system_id STRING,
    raw_payload_json STRING,
    ingestion_timestamp TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

# Use Memory Stream for simulation
memory_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

# Simulate FHIR Payload generation
simulated_fhir = memory_stream \
    .withColumn("payload_id", col("value").cast("string")) \
    .withColumn("source_system_id", lit("FHIR_R4_Sim")) \
    .withColumn("raw_payload_json", lit('{"resourceType": "Observation", "status": "final"}')) \
    .withColumn("ingestion_timestamp", current_timestamp())

checkpoint_dir = tempfile.mkdtemp()

# Write Stream to Iceberg
query = simulated_fhir.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .option("path", "local.ehdip_bronze_db.raw_payloads") \
    .option("checkpointLocation", checkpoint_dir) \
    .start()

import time
time.sleep(10) # Run for 10 seconds
query.stop()
print("Stream processed and stopped.")

# Verify Data
spark.sql("SELECT * FROM local.ehdip_bronze_db.raw_payloads LIMIT 5").show(truncate=False)